# 145 — Agentes de navegador

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

La solución valida el contrato mínimo sin asumir un valor interno específico.


In [ ]:
result = run_lab("agent", seed=145)
assert result["kind"] == "agent"
assert result["evidence"]
show(result)


## Solución 1 — Canal por subtarea

- (a) **DOM/AX**: campos con roles y nombres estándar; `fill` es exacto.
- (b) **Visión**: el canvas no expone semántica; solo los píxeles contienen
  el precio.
- (c) **DOM/AX**: el estado `disabled` es un atributo/estado de
  accesibilidad; visualmente un gris puede engañar.
- (d) **Híbrido**: el DOM dice que ambos botones existen; solo la imagen
  revela la superposición real y cuál recibe el clic.
- (e) **DOM/AX**: extraer texto estructurado de una tabla es trivial en el
  árbol y carísimo (y propenso a error de OCR) por visión.


## Solución 2 — Generalización de la cuenta

- (a) Visión: `0.99⁴·0.9² = 0.961·0.81 ≈ 0.778`. DOM: `0.99³ ≈ 0.970`. ✔
- (b) Se necesita `0.961·p² = 0.970` ⇒ `p² = 1.010` ⇒ **imposible**: ni con
  grounding perfecto (p=1: 0.961) alcanza al DOM, porque hace más acciones.
  Igualar exigiría también recortar pasos.
- (c) Con reintento: `p' = 0.9 + 0.1·0.9 = 0.99` ⇒ visión ≈ `0.99⁶ ≈ 0.941`.
  El DOM (0.970) sigue ganando, pero la brecha se reduce de 19 a 3 puntos: la
  verificación barata compensa la mayoría del déficit de grounding.


In [ ]:
vision = 0.99**4 * 0.9**2
dom = 0.99**3
vision_retry = 0.99**4 * (0.9 + 0.1*0.9)**2
print(f"vision={vision:.3f} dom={dom:.3f} vision+retry={vision_retry:.3f}")


## Solución 3 — Verificación programática

- (a) `assert api.get_profile().language == "es"` — el screenshot final
  podría mostrar el menú abierto en español sin haberse guardado el cambio.
- (b) `assert comment_id not in api.get_comments(post_id)` — la imagen puede
  mostrar el comentario "desaparecido" por un filtro de vista, no borrado.
- (c) `assert exists(download_dir / "informe_mensual.csv") and
  filas(csv) > 0` — el screenshot muestra el clic en "Descargar", no que el
  fichero llegó completo y no vacío.

Patrón común: verificar el **estado del sistema**, no la apariencia del
último instante — exactamente el criterio de WebArena.


## Solución 4 — Mapeo del contrato al bucle

- `kind`/semilla → configuración reproducible del episodio (la "pestaña" y la
  tarea).
- decisiones/`evidence` → el historial acción-observación: lo que un agente
  de navegador registraría como trazas.
- `limitations` → condiciones de salida honestas: lo que el episodio no
  garantiza.

La pieza ausente: **el entorno adversario asíncrono** — re-renders,
latencias variables, contenido no confiable que intenta inyectar
instrucciones. El lab es determinista y benigno; esa pieza cubre justamente
los riesgos (estado obsoleto, inyección, bucles) que separan la demo del
despliegue, y es la razón de ser de la clase 147.


In [ ]:
from ai_evolution.labs import run_lab

result = run_lab("agent", seed=145)
assert result["kind"] == "agent"
print("evidencias:", len(result["evidence"]))
print("limitaciones:", result["limitations"])
